In [ ]:
# Install dependencies
pip install vllm huggingface_hub[cli] python-multipart matplotlib datasets seaborn

# Start vLLM servers
python run_replicas.py --host 127.0.0.1 --worker-ports 8001,8002,8003,8004 --gpu-indices 0,1,2,3 --model-name "Qwen/Qwen2.5-1.5B-Instruct" --enable-prefix-caching

# Run single benchmark
## Start server
python server.py --worker-urls http://localhost:8001 http://localhost:8002 http://localhost:8003 http://localhost:8004 --host 127.0.0.1 --port 8000 --policy round_robin

## Then, run either generated-shared-prefix or sharegpt benchmark
python benchmark.py --backend vllm --host 127.0.0.1 --port 8000 --model Qwen/Qwen2.5-1.5B-Instruct --dataset-name generated-shared-prefix --gen-num-groups 1 --gen-prompts-per-group 1 --gen-system-prompt-len 2048 --gen-question-len 128 --gen-output-len 256 
python benchmark.py --backend vllm --host 127.0.0.1 --port 8000 --model Qwen/Qwen2.5-1.5B-Instruct --dataset-name sharegpt --dataset-path /home/ray/default/work/ray/prefix_aware_playground/current_work/sharegpt.json --num-prompts 1 --sharegpt-max-conversations 800 --max-concurrency 128

# Run benchmark sweep (will start servers with different policiesautomatically)
python sweep_strategies.py

# Visualize results
python display_prefix_sweep.ipynb